# Problema: Duplicatas e Múltiplos Registros no GCN

Este notebook demonstra o problema de duplicatas e múltiplos registros
no pipeline NASA GCN, especialmente para alertas de ondas gravitacionais.

## Contexto
- O GCN (Gamma-ray Coordinates Network) envia alertas de eventos astronômicos
- Um mesmo evento pode gerar MÚLTIPLOS alertas (PRELIMINARY → INITIAL → UPDATE)
- Sem tratamento adequado, temos "duplicatas semânticas" na Silver layer

## 1. Configuração

In [0]:
# Configurar catálogo (ajuste conforme seu ambiente)
CATALOG = "sandbox"  # ou "nasa_gcn" para prod
SCHEMA_SILVER = "silver"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_SILVER}")

## 2. Análise: Alertas de Ondas Gravitacionais (GW Alerts)

Vamos analisar a tabela `gcn_gwalert` que contém alertas do LIGO/Virgo/KAGRA.

In [0]:
# Total de registros na tabela
total_alerts = spark.sql("SELECT COUNT(*) as total FROM gcn_gwalert").collect()[0]["total"]
print(f"Total de alertas GW na Silver: {total_alerts:,}")

In [0]:
# Quantos eventos ÚNICOS temos?
unique_events = spark.sql("""
    SELECT COUNT(DISTINCT event_id) as unique_events
    FROM gcn_gwalert
    WHERE event_id IS NOT NULL
""").collect()[0]["unique_events"]

print(f"Eventos únicos: {unique_events:,}")
print(f"Média de alertas por evento: {total_alerts / unique_events:.1f}")

### Problema Identificado

Cada evento gravitacional gera múltiplos alertas com diferentes `alert_type`:
- `PRELIMINARY` - Detecção inicial automática
- `INITIAL` - Primeira análise humana
- `UPDATE` - Atualizações com mais dados
- `RETRACTION` - Cancelamento (falso positivo)

In [0]:
# Distribuição de tipos de alerta
display(spark.sql("""
    SELECT
        alert_type,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM gcn_gwalert
    GROUP BY alert_type
    ORDER BY count DESC
"""))

### Exemplo: Eventos com Múltiplos Alertas

In [0]:
# Top 10 eventos com mais alertas
display(spark.sql("""
    SELECT
        event_id,
        COUNT(*) as alert_count,
        COLLECT_SET(alert_type) as alert_types,
        MIN(kafka_timestamp) as first_alert,
        MAX(kafka_timestamp) as last_alert,
        TIMESTAMPDIFF(MINUTE, MIN(kafka_timestamp), MAX(kafka_timestamp)) as duration_minutes
    FROM gcn_gwalert
    WHERE event_id IS NOT NULL
    GROUP BY event_id
    HAVING COUNT(*) > 1
    ORDER BY alert_count DESC
    LIMIT 10
"""))

### Visualização: Timeline de um Evento Específico

In [0]:
# Selecionar um evento com múltiplos alertas para análise detalhada
sample_event = spark.sql("""
    SELECT event_id
    FROM (
        SELECT event_id, COUNT(*) as alert_count
        FROM gcn_gwalert
        WHERE event_id IS NOT NULL
        GROUP BY event_id
    )
    WHERE alert_count > 3
    ORDER BY alert_count DESC
    LIMIT 1
""").collect()

if sample_event:
    event_id = sample_event[0]["event_id"]
    print(f"Analisando evento: {event_id}")

    display(spark.sql(f"""
        SELECT
            event_id,
            alert_type,
            kafka_timestamp,
            ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY kafka_timestamp) as sequence
        FROM gcn_gwalert
        WHERE event_id = '{event_id}'
        ORDER BY kafka_timestamp
    """))
else:
    print("Nenhum evento com múltiplos alertas encontrado")

## 3. Análise: GCN Circulars

Circulares científicas também podem ter múltiplas versões para o mesmo evento.

In [0]:
# Eventos com múltiplas circulares
display(spark.sql("""
    SELECT
        event_id,
        COUNT(*) as circular_count,
        COLLECT_LIST(circular_id) as circular_ids
    FROM gcn_circulars
    WHERE event_id IS NOT NULL
    GROUP BY event_id
    HAVING COUNT(*) > 5
    ORDER BY circular_count DESC
    LIMIT 10
"""))

## 4. Impacto na Camada Gold

Sem deduplicação, as agregações podem ficar incorretas.

In [0]:
# Verificar a tabela gold de eventos
display(spark.sql(f"""
    SELECT
        event_id,
        circular_count,
        alert_type,
        last_updated
    FROM {CATALOG}.gold.gcn_events_summary
    ORDER BY circular_count DESC
    LIMIT 10
"""))

## 5. O Problema Resumido

| Cenário | Problema | Impacto |
|---------|----------|---------|
| GW Alerts | Múltiplos alertas por evento | Contagens infladas, estado inconsistente |
| Circulars | Múltiplas circulares por evento | OK (esperado - são documentos diferentes) |
| Notices | Possíveis duplicatas de mensagens | Dados redundantes |

### Solução: AUTO CDC

O próximo notebook demonstra como usar **AUTO CDC** para:
- **SCD Type 1**: Manter apenas o registro mais recente (última versão)
- **SCD Type 2**: Manter histórico completo com timestamps de validade

## Próximo Passo

Veja o notebook `2_solucao_auto_cdc` para a implementação da solução.